<a href="https://colab.research.google.com/github/guyberelzon-rgb/nlp-sentiment-analysis/blob/main/nlp_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Mining Excersize - NLP Sentiment Analysis

Guy Berelzon &
Ariel Sobol

In [ ]:
# Libraries
import pandas as pd
import numpy as np

from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/text_training.csv')
df.head()


Mounted at /content/drive


,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick,rating
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Data splitting and validation strategies
from sklearn.model_selection import (train_test_split, KFold,
                                     StratifiedKFold, cross_val_score,
                                     cross_val_predict)

# Text feature extraction
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Text feature extraction
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.pipeline import Pipeline

# Strategy For Analysis

The sentiment analysis will be 0 - bad review, 1 - good review

For each type of train test, we will check different types of models in order to find the best combinations for the model.

The strategies for dividing the data into train and test:

* 80/20 - Stratified
* K-Folds: Splits randomly, so some folds might end up with an imbalanced class distribution (one fold has 90% positive reviews).
* Stratified K-Folds (not taught in the course) This system ensures each fold has roughly the same positive/negative ratio as the full dataset. With a small dataset of 2000 reviews, a bad random split could give misleading scores. This shows a real world example split.

The Models we will check for the text mining

* Logistic Regression (Base) - L2 regularization, default C = 1.0. The standard linear classifier

*  Logistic Regression with L1 regularization (l1 Lasso). C tuned across multiple values. This model drives weak features to exactly zero, an automatic feature selection, so any feature that stayed non-zero is a feature the model decided to keep.

* Logistic Regression with L2 regularization (Ridge Regression). C tuned across multiple values, shrinks all features but keeps them all.

*  Random Forest - Decision trees using bagging. Used to handle high-dimensional data

*  Gradient Boosting - Each tree correcting the errors of the previous one.

*  XGBoosting - An optimized implementation of Gradient Boosting, stronger and faster.

* Support Vector Machine - SVM (Linear). Finds the maximum margin boundary between positive and negative reviews. Strong performer on high dimensional sparse text data.

In the course we learned several metrics for evaluating classification models: Accuracy, Precision, Recall, F1 and AUC. These will be used to decide upon the best model.




# Phase 1 : Summary of the Database

This part is used to understand the dataset we will be using for the analysis of the text mining.


In [ ]:

#Table Size
print("Number of records (rows):", df.shape[0])
print("Number of features (columns):", df.shape[1])
# Types of Reviews Distribution
print("Rating distribution:")
print(df['rating'].value_counts())

#Top words in the dataset
top_words_overall = df.drop(['ID', 'rating'], axis=1).sum().sort_values(ascending=False).head(10)
print("Top 10 words overall:\n", top_words_overall, "\n")

top_words_positive = df[df['rating'] == 1].drop(['ID', 'rating'], axis=1).sum().sort_values(ascending=False).head(10)
print("Top 10 words in POSITIVE reviews:\n", top_words_positive, "\n")

top_words_negative = df[df['rating'] == 0].drop(['ID', 'rating'], axis=1).sum().sort_values(ascending=False).head(10)
print("Top 10 words in NEGATIVE reviews:\n", top_words_negative, "\n")

# Words most associated with POSITIVE vs NEGATIVE reviews
word_counts = df.drop(['ID', 'rating'], axis=1)
pos_mean = word_counts[df['rating'] == 1].mean()
neg_mean = word_counts[df['rating'] == 0].mean()
diff = (pos_mean - neg_mean).sort_values(ascending=False)

print("Most discriminative POSITIVE words:\n", diff.head(10))
print("Most discriminative NEGATIVE words:\n", diff.tail(10))

# Average Words per Review Type
avg_words_per_review = (df.drop(['ID', 'rating'], axis=1) > 0).sum(axis=1).mean()
print(f"Average number of unique words per review: {avg_words_per_review:.2f}\n")

# Sparsity Matrix
total_elements = df.drop(['ID', 'rating'], axis=1).size
total_zeros = (df.drop(['ID', 'rating'], axis=1) == 0).sum().sum()
sparsity = (total_zeros / total_elements) * 100
print(f"Matrix Sparsity (percentage of zeros): {sparsity:.2f}%")

print(df.groupby('rating')['rating'].count())

review_lengths = (df.drop(['ID', 'rating'], axis=1) > 0).sum(axis=1)
print(review_lengths.groupby(df['rating']).mean())

Number of records (rows): 2000
Number of features (columns): 2002
Rating distribution:
rating
1    1243
0     757
Name: count, dtype: int64
Top 10 words overall:
 the     7288
and     5038
this    2709
for     1979
that    1717
have    1393
was     1343
not     1313
with    1304
but     1292
dtype: int64 

Top 10 words in POSITIVE reviews:
 the     4090
and     3211
this    1595
for     1301
that     979
have     875
with     821
you      802
are      798
but      713
dtype: int64 

Top 10 words in NEGATIVE reviews:
 the     3198
and     1827
this    1114
was      786
that     738
not      693
for      678
but      579
have     518
they     491
dtype: int64 

Most discriminative POSITIVE words:
 love     0.272347
great    0.264561
are      0.171718
and      0.169792
you      0.164368
for      0.151021
tea      0.142374
best     0.139149
good     0.111206
make     0.103003
dtype: float64
Most discriminative NEGATIVE words:
 were      -0.169707
tast      -0.184145
that      -0.187290
wou

### Summary of Dataset

### Rating
1) 62% positive rating (1) - 1243 positive ratings
2) 38% negative rating (0) - 757 negative ratings

The discriminative word means is calculated by the average times a word appears in a positive review  minus the average times a word appears in a negative review. If a word appears more times in a positive review, than the pos_mean is bigger then the difference is positive (diff = pos_mean - neg_mean). If a word appears more in a negative review, than the difference i negative.

For example, the word love appears 0.272 more times in a positive review than in a negative review. These words simply show up more in reviews where the customer was happy so it is a strong signal.

The word "the" appears 0.934 times more on negative reviews than in positive ones. This is interesting, since the word "the" is not a negative or positive word. This parameter means negative reviewers tend to write longer more detailed complaints since they use the word "the" more often in a review, for example: "The product broke" , "the tast was awful", "the packaging was damaged". The words product and tast also show up more often in negative than positive reviews.

To make sure that statistically negative reviews are longer than positive reviews, we checked the average number of words per review. Positive reviews have 43 words, and negative reviews have 46 words. This points out that unhappy customers write more.

Sparsity 97.79% is expected and normal for a words matrix. Most reviews only use a small group of words out of the full vocabulary, that is the reason that a vast majority of words in the word matrix of reviews have a very minimal or zero counts of words. This is ok.

### Further understanding the words and what they may predict

This part measures how useful each word is for predicting the rating. The ig_scores (information gain) helps us answer the question, if a word appears in a review, how much does that tell me about whether the review is positive or negative. The closwer the score is to 0, tells you almost nothing about the rating it should be part of. The word appears in positive and negative reviews. If the score is close to 1, that means the word appears almost exclusively in one type of review.

In [ ]:


X = df.drop(columns=['ID', 'rating'])
y = df['rating']

# Information Gain Classification
ig_scores = mutual_info_classif(X, y, random_state=42)
ig_series = pd.Series(ig_scores, index=X.columns).sort_values(ascending=False)
print("Done!")
print(ig_series.head(10))

Done!
great     0.035698
not       0.035541
was       0.035373
vendor    0.034802
the       0.033647
hearti    0.033132
third     0.032211
rancid    0.031262
milk      0.029764
guy       0.029441
dtype: float64


The scores are all very low (around 0.03), which is expected for bag-of-words text data with 2000 features. No single word is highly predictive on its own — sentiment is expressed through combinations of words, not one word alone. The words "great" and "not" top the list, which makes sense. "great" appears overwhelmingly in positive reviews, while "not" is a strong negation word that flips sentiment ("not good", "not fresh", "not what I expected"), making it very useful for identifying negative reviews.

Since all scores are low and no single word dominates, feature selection requires care. Keeping too few features risks losing useful signal, while keeping too many adds noise that confuses the model. In this analysis we pass all 2000 features to every model — with only 2000 samples, removing features risks discarding words that carry small but real signal.

Instead of explicit feature selection, we rely on two implicit mechanisms. First, TF-IDF down-weights words that appear frequently across all reviews and up-weights words that are specific and informative. Second, L2 regularization in the best performing model shrinks the weights of irrelevant words towards zero without removing them entirely. This combination means the model effectively learns which words matter and which don't, without us deciding which features to keep or discard.

## Cleaning the Data



In [ ]:
# There are no missing values and no negative counts
print(df.isnull().sum().sum())
print((df.drop(columns=['ID','rating']) < 0).sum().sum())

0
0


# Strategy 1 - Dividing the data set into train (80%) and Test (20%)

In [ ]:


# ── Step 1: Define X and y ───────────────────────────────────────────────────
X = df.drop(columns=['ID', 'rating'])
y = df['rating']

# ── Step 2: Split (stratified — preserves 62%/38% class ratio) ──────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)}")
print(f"Train class ratio: {y_train.mean():.2%} positive")
print(f"Val   class ratio: {y_val.mean():.2%} positive\n")

# ── Step 3: TF-IDF (fit on train only, transform both) ──────────────────────
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train)
X_val_tfidf   = tfidf_transformer.transform(X_val)

# ── Step 4: Base Models ──────────────────────────────────────────────────────
base_models = {
    "Logistic Regression (Base)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest (Base)":       RandomForestClassifier(random_state=42),
    "Gradient Boosting (Base)":   GradientBoostingClassifier(random_state=42),
    "XGBoost (Base)":             XGBClassifier(random_state=42, eval_metric='logloss'),
    "SVM (Linear)":               CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42))
}

results = {}

for name, model in base_models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_val_tfidf)
    y_prob = model.predict_proba(X_val_tfidf)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    results[name] = acc

    print(f"--- {name} ---")
    print(f"Accuracy:         {acc:.4f}")
    print(f"Precision:        {precision_score(y_val, y_pred):.4f}")
    print(f"Recall:           {recall_score(y_val, y_pred):.4f}")
    print(f"F1 Score:         {f1_score(y_val, y_pred):.4f}")
    print(f"AUC:              {roc_auc_score(y_val, y_prob):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_val, y_pred)}\n")

# ── Step 5: L1 Logistic Regression — C Tuning ───────────────────────────────
print("--- L1 Logistic Regression - C Tuning ---")
best_c_l1, best_acc_l1, best_model_l1 = None, 0, None

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    model = LogisticRegression(penalty='l1', solver='liblinear',
                               C=c, max_iter=1000, random_state=42)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_val_tfidf)
    y_prob = model.predict_proba(X_val_tfidf)[:, 1]
    features_kept = np.sum(model.coef_[0] != 0)
    acc = accuracy_score(y_val, y_pred)
    print(f"C={c:<5} | Features: {features_kept:<4} | Accuracy: {acc:.4f} | "
          f"F1: {f1_score(y_val, y_pred):.4f} | AUC: {roc_auc_score(y_val, y_prob):.4f}")

    if acc > best_acc_l1:
        best_acc_l1, best_c_l1, best_model_l1 = acc, c, model

y_pred_l1 = best_model_l1.predict(X_val_tfidf)
y_prob_l1 = best_model_l1.predict_proba(X_val_tfidf)[:, 1]
results[f"LR L1 (C={best_c_l1})"] = best_acc_l1

print(f"\n--- Best L1 Model: C={best_c_l1} ---")
print(f"Accuracy:         {best_acc_l1:.4f}")
print(f"Precision:        {precision_score(y_val, y_pred_l1):.4f}")
print(f"Recall:           {recall_score(y_val, y_pred_l1):.4f}")
print(f"F1 Score:         {f1_score(y_val, y_pred_l1):.4f}")
print(f"AUC:              {roc_auc_score(y_val, y_prob_l1):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_val, y_pred_l1)}\n")

# ── Step 6: L2 Logistic Regression (Ridge) — C Tuning ───────────────────────
print("--- L2 Logistic Regression (Ridge) - C Tuning ---")
best_c_l2, best_acc_l2, best_model_l2 = None, 0, None

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    model = LogisticRegression(penalty='l2', solver='liblinear',
                               C=c, max_iter=1000, random_state=42)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_val_tfidf)
    y_prob = model.predict_proba(X_val_tfidf)[:, 1]
    acc = accuracy_score(y_val, y_pred)
    print(f"C={c:<5} | Accuracy: {acc:.4f} | "
          f"F1: {f1_score(y_val, y_pred):.4f} | AUC: {roc_auc_score(y_val, y_prob):.4f}")

    if acc > best_acc_l2:
        best_acc_l2, best_c_l2, best_model_l2 = acc, c, model

y_pred_l2 = best_model_l2.predict(X_val_tfidf)
y_prob_l2 = best_model_l2.predict_proba(X_val_tfidf)[:, 1]
results[f"LR L2 (C={best_c_l2})"] = best_acc_l2

print(f"\n--- Best L2 Model: C={best_c_l2} ---")
print(f"Accuracy:         {best_acc_l2:.4f}")
print(f"Precision:        {precision_score(y_val, y_pred_l2):.4f}")
print(f"Recall:           {recall_score(y_val, y_pred_l2):.4f}")
print(f"F1 Score:         {f1_score(y_val, y_pred_l2):.4f}")
print(f"AUC:              {roc_auc_score(y_val, y_prob_l2):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_val, y_pred_l2)}\n")

# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 45)
print("STRATEGY 1 SUMMARY — 80/20 Stratified Split")
print("=" * 45)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<30} Accuracy: {acc:.4f}")
print("=" * 45)

Train size: 1600 | Val size: 400
Train class ratio: 62.12% positive
Val   class ratio: 62.25% positive

--- Logistic Regression (Base) ---
Accuracy:         0.8275
Precision:        0.8125
Recall:           0.9398
F1 Score:         0.8715
AUC:              0.9210
Confusion Matrix:
[[ 97  54]
 [ 15 234]]

--- Random Forest (Base) ---
Accuracy:         0.8225
Precision:        0.8134
Recall:           0.9277
F1 Score:         0.8668
AUC:              0.9069
Confusion Matrix:
[[ 98  53]
 [ 18 231]]

--- Gradient Boosting (Base) ---
Accuracy:         0.7900
Precision:        0.7835
Recall:           0.9157
F1 Score:         0.8444
AUC:              0.8824
Confusion Matrix:
[[ 88  63]
 [ 21 228]]

--- XGBoost (Base) ---
Accuracy:         0.8225
Precision:        0.8371
Recall:           0.8876
F1 Score:         0.8616
AUC:              0.8919
Confusion Matrix:
[[108  43]
 [ 28 221]]

--- SVM (Linear) ---
Accuracy:         0.8575
Precision:        0.8721
Recall:           0.9036
F1 Score:   

## Strategy 1 — 80/20 Stratified Split Results

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| **LR L2, C=7.0** | **0.8650** | **0.8707** | 0.9197 | **0.8945** | 0.9233 |
| SVM (Linear) | 0.8575 | 0.8721 | 0.9036 | 0.8876 | 0.9220 |
| LR L1, C=100.0 | 0.8450 | 0.8582 | 0.8996 | 0.8784 | 0.9109 |
| Logistic Regression (Base) | 0.8275 | 0.8125 | **0.9398** | 0.8715 | **0.9239** |
| Random Forest | 0.8225 | 0.8134 | 0.9277 | 0.8668 | 0.9069 |
| XGBoost | 0.8225 | 0.8371 | 0.8876 | 0.8616 | 0.8919 |
| Gradient Boosting | 0.7900 | 0.7835 | 0.9157 | 0.8444 | 0.8824 |

---

## Confusion Matrices

**Logistic Regression (Base)** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 97 (TN)           | 54 (FP)           |
| **Actual Positive** | 15 (FN)           | 234 (TP)          |

331 correct · 69 wrong

---

**Random Forest (Base)** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 98 (TN)           | 53 (FP)           |
| **Actual Positive** | 18 (FN)           | 231 (TP)          |

329 correct · 71 wrong

---

**Gradient Boosting (Base)** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 88 (TN)           | 63 (FP)           |
| **Actual Positive** | 21 (FN)           | 228 (TP)          |

316 correct · 84 wrong

---

**XGBoost (Base)** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 108 (TN)          | 43 (FP)           |
| **Actual Positive** | 28 (FN)           | 221 (TP)          |

329 correct · 71 wrong

---

**SVM (Linear)** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 118 (TN)          | 33 (FP)           |
| **Actual Positive** | 24 (FN)           | 225 (TP)          |

343 correct · 57 wrong

---

**LR L1, C=100.0** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 114 (TN)          | 37 (FP)           |
| **Actual Positive** | 25 (FN)           | 224 (TP)          |

338 correct · 62 wrong

---

**Best Model — LR L2 (Ridge), C=7.0** (400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 117 (TN)          | 34 (FP)           |
| **Actual Positive** | 20 (FN)           | 229 (TP)          |

346 correct · 54 wrong

In the 80/20 test we still used stratify = y, that means the test samples always hae exactly the same 62/38 ratio as the full dataset (real world). This creates a controlled test on 400 samples - the result is not influenced by a lucky or unlucky split (that would have happened without the stratify). Both test and train sets reflect the real world distribution. Stratify ensures that a random split that gives the training set 70% positive reviews does not happen. 80/20 is about the size of the train and test. Stratify is about balance of each set.


### The results For the 80/20 Split

In strategy 1, the 80/20 split, the best model is LR L2 (Ridge Regression) with C = 7.0 with an accuracy of 0.8650 and F1 of 0.8945, on a test sample of 400 reviews. The model correctly classified 346 reviews and made 54 mistakes.

The base model had the highest Recall and AUC in the strategy, that means the base model is very good at catching positive reviews but at the cost of more FPs. The SVM Model came second overall with the fewest errors among base models, showing that linear margin based models handle the text data well, as taught in class. The Tree-based models underperformed, as expected.

The L1 tuning improved the base model but was outperformed by L2. The L1 aggressively removed 1392 words (C = 100, Features = 608, Accuracy 0.8450). L2 Kept all 2000 features with reduced weights, allowing the model to capture the subtle signal spread across many words , this is better for sentiment analysis of text.




# Strategy 2 - K-Fold Cross-Validation

In [ ]:
warnings.filterwarnings('ignore')

# Step 1: Define X and y (full dataset)
X = df.drop(columns=['ID', 'rating'])
y = df['rating']

# Step 2: Find Best K
print("--- Finding Best K ---")
print(f"{'Model':<25} | {'K':<5} | {'Mean Accuracy':<15} | {'Std Dev':<10}")
print("-" * 65)

best_k, best_k_acc = 10, 0

models_k = {
    "Logistic Regression": Pipeline([('tfidf', TfidfTransformer()),
                                     ('clf',   LogisticRegression(max_iter=1000, random_state=42))]),
    "Random Forest":       Pipeline([('tfidf', TfidfTransformer()),
                                     ('clf',   RandomForestClassifier(random_state=42))])
}

for model_name, model in models_k.items():
    for k in [3, 5, 10]:
        kf = KFold(n_splits=k, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
        mean_acc = np.mean(scores)
        print(f"{model_name:<25} | {k:<5} | {mean_acc:.4f}          | {np.std(scores):.4f}")

        if model_name == "Logistic Regression" and mean_acc > best_k_acc:
            best_k_acc, best_k = mean_acc, k

print(f"\n  ★ Best K = {best_k} (Accuracy: {best_k_acc:.4f})\n")

# Step 3: Base Models (full metrics with best K)
kf = KFold(n_splits=best_k, shuffle=True, random_state=42)
results = {}

def full_eval(name, pipeline):
    y_pred = cross_val_predict(pipeline, X, y, cv=kf)
    y_prob = cross_val_predict(pipeline, X, y, cv=kf, method='predict_proba')[:, 1]

    acc = accuracy_score(y, y_pred)
    results[name] = acc

    print(f"--- {name} ---")
    print(f"Accuracy:         {acc:.4f}")
    print(f"Precision:        {precision_score(y, y_pred):.4f}")
    print(f"Recall:           {recall_score(y, y_pred):.4f}")
    print(f"F1 Score:         {f1_score(y, y_pred):.4f}")
    print(f"AUC:              {roc_auc_score(y, y_prob):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y, y_pred)}\n")

full_eval("Logistic Regression (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
]))

full_eval("Random Forest (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   RandomForestClassifier(random_state=42))
]))

full_eval("Gradient Boosting (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   GradientBoostingClassifier(random_state=42))
]))

full_eval("XGBoost (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   XGBClassifier(random_state=42, eval_metric='logloss'))
]))

full_eval("SVM (Linear)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42)))
]))

# Step 4: L1 Logistic Regression — C Tuning
print("--- L1 Logistic Regression - C Tuning ---")
best_c_l1, best_acc_l1 = None, 0

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    pipeline = Pipeline([('tfidf', TfidfTransformer()),
                         ('clf',   LogisticRegression(penalty='l1', solver='liblinear',
                                                      C=c, max_iter=1000, random_state=42))])
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring='accuracy')
    mean_acc = np.mean(scores)
    print(f"C={c:<5} | Accuracy: {mean_acc:.4f} ± {np.std(scores):.4f}")

    if mean_acc > best_acc_l1:
        best_acc_l1, best_c_l1 = mean_acc, c

print(f"\n--- Best L1 Model: C={best_c_l1} ---")
full_eval(f"LR L1 (C={best_c_l1})", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(penalty='l1', solver='liblinear',
                                 C=best_c_l1, max_iter=1000, random_state=42))
]))

# Step 5: L2 Logistic Regression (Ridge) — C Tuning
print("--- L2 Logistic Regression (Ridge) - C Tuning ---")
best_c_l2, best_acc_l2 = None, 0

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    pipeline = Pipeline([('tfidf', TfidfTransformer()),
                         ('clf',   LogisticRegression(penalty='l2', solver='liblinear',
                                                      C=c, max_iter=1000, random_state=42))])
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring='accuracy')
    mean_acc = np.mean(scores)
    print(f"C={c:<5} | Accuracy: {mean_acc:.4f} ± {np.std(scores):.4f}")

    if mean_acc > best_acc_l2:
        best_acc_l2, best_c_l2 = mean_acc, c

print(f"\n--- Best L2 Model: C={best_c_l2} ---")
full_eval(f"LR L2 (C={best_c_l2})", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(penalty='l2', solver='liblinear',
                                 C=best_c_l2, max_iter=1000, random_state=42))
]))

#  Summary ──
print("=" * 50)
print(f"STRATEGY 2 SUMMARY — {best_k}-Fold CV (non-stratified)")
print("=" * 50)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<30} Accuracy: {acc:.4f}")
print("=" * 50)

--- Finding Best K ---
Model                     | K     | Mean Accuracy   | Std Dev   
-----------------------------------------------------------------
Logistic Regression       | 3     | 0.8315          | 0.0285
Logistic Regression       | 5     | 0.8375          | 0.0268
Logistic Regression       | 10    | 0.8400          | 0.0311
Random Forest             | 3     | 0.8200          | 0.0220
Random Forest             | 5     | 0.8245          | 0.0193
Random Forest             | 10    | 0.8295          | 0.0328

  ★ Best K = 10 (Accuracy: 0.8400)

--- Logistic Regression (Base) ---
Accuracy:         0.8400
Precision:        0.8239
Recall:           0.9445
F1 Score:         0.8801
AUC:              0.9320
Confusion Matrix:
[[ 506  251]
 [  69 1174]]

--- Random Forest (Base) ---
Accuracy:         0.8295
Precision:        0.8141
Recall:           0.9405
F1 Score:         0.8727
AUC:              0.9149
Confusion Matrix:
[[ 490  267]
 [  74 1169]]

--- Gradient Boosting (Base) ---
Accu

# Strategy 2 — K-Fold (K=10, non-stratified)

## Results

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| **LR L2, C=7.0** | **0.8595** | 0.8638 | 0.9187 | **0.8904** | 0.9328 |
| SVM (Linear) | 0.8575 | **0.8748** | 0.8994 | 0.8869 | 0.9305 |
| LR L1, C=5.0 | 0.8475 | 0.8687 | 0.8890 | 0.8787 | 0.9210 |
| Logistic Regression (Base) | 0.8400 | 0.8239 | **0.9445** | 0.8801 | 0.9320 |
| XGBoost | 0.8375 | 0.8441 | 0.9059 | 0.8739 | 0.9087 |
| Random Forest | 0.8295 | 0.8141 | 0.9405 | 0.8727 | 0.9149 |
| Gradient Boosting | 0.8130 | 0.8041 | 0.9244 | 0.8600 | 0.8925 |

---

## Confusion Matrices

**Logistic Regression (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 506 (TN)          | 251 (FP)          |
| **Actual Positive** | 69 (FN)           | 1174 (TP)         |

1680 correct · 320 wrong

---

**Random Forest (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 490 (TN)          | 267 (FP)          |
| **Actual Positive** | 74 (FN)           | 1169 (TP)         |

1659 correct · 341 wrong

---

**Gradient Boosting (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 477 (TN)          | 280 (FP)          |
| **Actual Positive** | 94 (FN)           | 1149 (TP)         |

1626 correct · 374 wrong

---

**XGBoost (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 549 (TN)          | 208 (FP)          |
| **Actual Positive** | 117 (FN)          | 1126 (TP)         |

1675 correct · 325 wrong

---

**SVM (Linear)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 597 (TN)          | 160 (FP)          |
| **Actual Positive** | 125 (FN)          | 1118 (TP)         |

1715 correct · 285 wrong

---

**LR L1, C=5.0** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 590 (TN)          | 167 (FP)          |
| **Actual Positive** | 138 (FN)          | 1105 (TP)         |

1695 correct · 305 wrong

---

**Best Model — LR L2 (Ridge), C=7.0** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 577 (TN)          | 180 (FP)          |
| **Actual Positive** | 101 (FN)          | 1142 (TP)         |

1719 correct · 281 wrong



The model is trained and tested across all 2000 reviews . Before running the full analysis, we tested what K is better between 3,5 and 10 using Logistic Regression and Random Forest. K = 10 gave the highest accuracy in both models and was selected as the best K.

The SVM model had the highest accuracy and Precision. It corretly identified positive reviews while making fewer FPs (False Positive errors) than the other base models. The tree based models again underperformed.

L2 with C = 7.0 won overall with an accuracy of 0.8595, F1 of 0.8904 and the highest AUC of 0.9328 in this strategy.  1719 were correctly classified. The low std shows stable performance across all 10 folds.

## Comparison Between Strategy 1 And Strategy 2

| | Strategy 1 | Strategy 2 |
|---|---|---|
| Method | 80/20 stratified split | K-Fold (K=10) |
| Best model | LR L2, C=7.0 | LR L2, C=7.0 |
| Accuracy | **0.8650** | 0.8595 |
| Precision | **0.8707** | 0.8638 |
| Recall | **0.9197** | 0.9187 |
| F1 | **0.8945** | 0.8904 |
| AUC | 0.9233 | **0.9328** |

---

## Confusion Matrices — Best Model (LR L2, C=7.0) per Strategy

**Strategy 1 — 80/20 stratified split** (LR L2, C=7.0 · 400 samples)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 117 (TN)          | 34 (FP)           |
| **Actual Positive** | 20 (FN)           | 229 (TP)          |

346 correct · 54 wrong

---

**Strategy 2 — K-Fold (K=10)** (LR L2, C=7.0 · 2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 577 (TN)          | 180 (FP)          |
| **Actual Positive** | 101 (FN)          | 1142 (TP)         |

1719 correct · 281 wrong

---



### Key Comparison Observations

Although the accuracy is higher in Strategy 1, the result is more trustworthy in Strategy 2. It is averaged across 10 different splits covering the full dataset rather than a single 400 sample test.
Strategy 2 has a higher Recall and AUC since K-Folds gives a more reliable estimate by training on all 2000 reviews.

In Both strategies, L2 outperformed L1. Ridge Regression handles the correlated bag of words features better than Lasso's sparse selection.
For L1, the best C shifted from 100 in strategy 1 to 5 in strategy 2. This makes sense since more training data means the model needs stronger regularization to avoid overfitting.

# Strategy 3 - StratifiedKFold

The reason for using Stratified Kfold (not taught in class) is to train the model by the real proportions that there are in real life.

Standard K-Fold splits the data randomly, it could produce folds with unequal distributions of positive and negative reviews that don't resemble the real world. Stratified K-Folds solves this and trains the model near the real world distribution by preserving the original 62/38 ratio. We belive that this is important in Text Mining and Classification where the imbalance affects how the model learns sentiment patterns. The goal is to make sure that the model is trained and evaluated under the same real world conditions.

Finding the best K is a setup step — we need to identify which K value produces the most stable and accurate folds before evaluating all models. Only Logistic Regression and Random Forest are used in this step because they are fast to train, and K is a property of the data rather than the model. The winning K applies equally to all models evaluated below.



In [ ]:

warnings.filterwarnings('ignore')

# Step 1: Define X and y (full dataset)
X = df.drop(columns=['ID', 'rating'])
y = df['rating']

print(f"Class distribution: {y.value_counts().to_dict()}")
print(f"Class ratio: {y.mean():.2%} positive\n")

# Step 2: Find Best K
print("--- Finding Best K (Stratified) ---")
print(f"{'Model':<25} | {'K':<5} | {'Mean Accuracy':<15} | {'Std Dev':<10}")
print("-" * 65)

best_k, best_k_acc = 10, 0

models_k = {
    "Logistic Regression": Pipeline([('tfidf', TfidfTransformer()),
                                     ('clf',   LogisticRegression(max_iter=1000, random_state=42))]),
    "Random Forest":       Pipeline([('tfidf', TfidfTransformer()),
                                     ('clf',   RandomForestClassifier(random_state=42))])
}

for model_name, model in models_k.items():
    for k in [3, 5, 10]:
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
        mean_acc = np.mean(scores)
        print(f"{model_name:<25} | {k:<5} | {mean_acc:.4f}          | {np.std(scores):.4f}")

        if model_name == "Logistic Regression" and mean_acc > best_k_acc:
            best_k_acc, best_k = mean_acc, k

print(f"\n  ★ Best K = {best_k} (Accuracy: {best_k_acc:.4f})\n")

# Step 3: Base Models (full metrics with best K)
skf = StratifiedKFold(n_splits=best_k, shuffle=True, random_state=42)
results = {}

def full_eval(name, pipeline):
    y_pred = cross_val_predict(pipeline, X, y, cv=skf)
    y_prob = cross_val_predict(pipeline, X, y, cv=skf, method='predict_proba')[:, 1]

    acc = accuracy_score(y, y_pred)
    results[name] = acc

    print(f"--- {name} ---")
    print(f"Accuracy:         {acc:.4f}")
    print(f"Precision:        {precision_score(y, y_pred):.4f}")
    print(f"Recall:           {recall_score(y, y_pred):.4f}")
    print(f"F1 Score:         {f1_score(y, y_pred):.4f}")
    print(f"AUC:              {roc_auc_score(y, y_prob):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y, y_pred)}\n")

full_eval("Logistic Regression (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(max_iter=1000, random_state=42))
]))

full_eval("Random Forest (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   RandomForestClassifier(random_state=42))
]))

full_eval("Gradient Boosting (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   GradientBoostingClassifier(random_state=42))
]))

full_eval("XGBoost (Base)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   XGBClassifier(random_state=42, eval_metric='logloss'))
]))

full_eval("SVM (Linear)", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42)))
]))

# Step 4: L1 Logistic Regression — C Tuning
print("--- L1 Logistic Regression - C Tuning ---")
best_c_l1, best_acc_l1 = None, 0

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    pipeline = Pipeline([('tfidf', TfidfTransformer()),
                         ('clf',   LogisticRegression(penalty='l1', solver='liblinear',
                                                      C=c, max_iter=1000, random_state=42))])
    scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    mean_acc = np.mean(scores)
    print(f"C={c:<5} | Accuracy: {mean_acc:.4f} ± {np.std(scores):.4f}")

    if mean_acc > best_acc_l1:
        best_acc_l1, best_c_l1 = mean_acc, c

print(f"\n--- Best L1 Model: C={best_c_l1} ---")
full_eval(f"LR L1 (C={best_c_l1})", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(penalty='l1', solver='liblinear',
                                 C=best_c_l1, max_iter=1000, random_state=42))
]))

# Step 5: L2 Logistic Regression (Ridge) — C Tuning
print("--- L2 Logistic Regression (Ridge) - C Tuning ---")
best_c_l2, best_acc_l2 = None, 0

for c in [0.1, 1.0, 5.0, 7.0, 10.0, 15.0, 20.0, 25.0, 30.0, 50.0, 100.0]:
    pipeline = Pipeline([('tfidf', TfidfTransformer()),
                         ('clf',   LogisticRegression(penalty='l2', solver='liblinear',
                                                      C=c, max_iter=1000, random_state=42))])
    scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    mean_acc = np.mean(scores)
    print(f"C={c:<5} | Accuracy: {mean_acc:.4f} ± {np.std(scores):.4f}")

    if mean_acc > best_acc_l2:
        best_acc_l2, best_c_l2 = mean_acc, c

print(f"\n--- Best L2 Model: C={best_c_l2} ---")
full_eval(f"LR L2 (C={best_c_l2})", Pipeline([
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(penalty='l2', solver='liblinear',
                                 C=best_c_l2, max_iter=1000, random_state=42))
]))

# Summary
print("=" * 55)
print(f"STRATEGY 3 SUMMARY — {best_k}-Fold Stratified CV")
print("=" * 55)
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<30} Accuracy: {acc:.4f}")
print("=" * 55)

Class distribution: {1: 1243, 0: 757}
Class ratio: 62.15% positive

--- Finding Best K (Stratified) ---
Model                     | K     | Mean Accuracy   | Std Dev   
-----------------------------------------------------------------
Logistic Regression       | 3     | 0.8255          | 0.0032
Logistic Regression       | 5     | 0.8350          | 0.0144
Logistic Regression       | 10    | 0.8435          | 0.0224
Random Forest             | 3     | 0.8260          | 0.0021
Random Forest             | 5     | 0.8350          | 0.0137
Random Forest             | 10    | 0.8330          | 0.0196

  ★ Best K = 10 (Accuracy: 0.8435)

--- Logistic Regression (Base) ---
Accuracy:         0.8435
Precision:        0.8261
Recall:           0.9477
F1 Score:         0.8827
AUC:              0.9308
Confusion Matrix:
[[ 509  248]
 [  65 1178]]

--- Random Forest (Base) ---
Accuracy:         0.8330
Precision:        0.8172
Recall:           0.9421
F1 Score:         0.8752
AUC:              0.9134
Co

## Strategy 3 — Stratified K-Fold (K=10) - Results

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| **LR L2, C=7.0** | **0.8560** | 0.8615 | 0.9155 | 0.8877 | 0.9320 |
| SVM (Linear) | 0.8530 | **0.8681** | 0.9002 | 0.8839 | 0.9300 |
| LR L1, C=7.0 | 0.8480 | 0.8659 | 0.8938 | 0.8797 | 0.9224 |
| Logistic Regression (Base) | 0.8435 | 0.8261 | **0.9477** | **0.8827** | **0.9308** |
| Random Forest | 0.8330 | 0.8172 | 0.9421 | 0.8752 | 0.9134 |
| XGBoost | 0.8320 | 0.8449 | 0.8938 | 0.8686 | 0.9045 |
| Gradient Boosting | 0.8140 | 0.8073 | **0.9204** | 0.8602 | 0.8960 |

---

## Confusion Matrices

**Logistic Regression (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 509 (TN)          | 248 (FP)          |
| **Actual Positive** | 65 (FN)           | 1178 (TP)         |

1687 correct · 313 wrong

---

**Random Forest (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 495 (TN)          | 262 (FP)          |
| **Actual Positive** | 72 (FN)           | 1171 (TP)         |

1666 correct · 334 wrong

---

**Gradient Boosting (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 484 (TN)          | 273 (FP)          |
| **Actual Positive** | 99 (FN)           | 1144 (TP)         |

1628 correct · 372 wrong

---

**XGBoost (Base)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 553 (TN)          | 204 (FP)          |
| **Actual Positive** | 132 (FN)          | 1111 (TP)         |

1664 correct · 336 wrong

---

**SVM (Linear)** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 587 (TN)          | 170 (FP)          |
| **Actual Positive** | 124 (FN)          | 1119 (TP)         |

1706 correct · 294 wrong

---

**LR L1, C=7.0** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 585 (TN)          | 172 (FP)          |
| **Actual Positive** | 132 (FN)          | 1111 (TP)         |

1696 correct · 304 wrong

---

**Best Model — LR L2 (Ridge), C=7.0** (2000 samples aggregated)

|                     | Predicted Negative | Predicted Positive |
|---------------------|-------------------|-------------------|
| **Actual Negative** | 574 (TN)          | 183 (FP)          |
| **Actual Positive** | 105 (FN)          | 1138 (TP)         |

1712 correct · 288 wrong

---

### Strategy 3 Results

Strategy 3 uses K = 10 with a std of 0.0224, lower than the regular K-Folds 0.0311 at the same K which means that the stratification produces more consistent results than the regular K-Folds. The stratified stratigy gurantees that every fold maintains the original 62/38 ratio of positive and negative reviews.

Among the base models, the Logistic Regression leads in accuracy and highest Recall. SVM Came second and again the tree based models underperformed.

For L1 tuning, the best C shifted to 7.0 and a lower std than all the L1 tuning in the other strategies. This points out that the stratification brings strong stability.

The L2 tuning with C = 7.0 won with 1712 correctly classified reviews with only 105 FNs and 183 FPs. Compared to Strategy 2, the accuracy is slightly lower but the std is lower too, and the balance iss guranteed in every fold. This makes strategy 3 the most reliable of the three.


---

## Strategy 2 vs Strategy 3 — Full Comparison

| | Strategy 2 | Strategy 3 |
|---|---|---|
| Method | K-Fold (K=10) | Stratified K-Fold (K=10) |
| Best model | LR L2, C=7.0 | LR L2, C=7.0 |
| Accuracy | **0.8595** | 0.8560 |
| Precision | **0.8638** | 0.8615 |
| Recall | **0.9187** | 0.9155 |
| F1 | **0.8904** | 0.8877 |
| AUC | 0.9328 | **0.9320** |

---

# Full Comparison — All 3 Strategies

| | Strategy 1 | Strategy 2 | Strategy 3 |
|---|---|---|---|
| Method | 80/20 stratified split | K-Fold (K=10) | Stratified K-Fold (K=10) |
| Best model | LR L2, C=7.0 | LR L2, C=7.0 | LR L2, C=7.0 |
| Accuracy | **0.8650** | 0.8595 | 0.8560 |
| Precision | **0.8707** | 0.8638 | 0.8615 |
| Recall | **0.9197** | 0.9187 | 0.9155 |
| F1 | **0.8945** | 0.8904 | 0.8877 |
| AUC | 0.9233 | **0.9328** | 0.9320 |



# Model Decision

**Strategy 3 — Stratified K-Fold with K=10, LR L2 (Ridge), C=7.0**

Across all three validation strategies, LR L2 (Ridge Regression) with C = 7.0 won.

Strategy 1 shows the highest results of Accuracy, Precision, Recall and F1, but it is tested on 400 reviews, a very small window. Although the stratification removes the luck factor, but training on 1600 samples and testing on 400 is very small.

Strategy 2 tests across all 2000 reviews and achieves the highest AUC, but it folds may have slightly unequal class ratios since it is not stratified.

Strategy 3 combines both advantages - all 2000 samples with guaranteed 62/38 ratio of positive and negative reviews per fold. This is the most trustworthy of the three.

Throughout all the strategies, L2 consistently outperformed L1 and SVM, confirming that keeping all features with weight shrinkage works better than sparse selection on this text dataset.

The low std proves that the Stratified K-Fold is more consistent, especially in Text mining where feature noise is high (bag-of-words has hundreds of irrelevant words), you want the most stable evaluation possible.


**Decision**

**LR L2 (Ridge), C = 7.0 trained on all 2000 samples, using Stratified K-Fold K=10.**

* **Accuracy: 0.8560**

* Precision: 0.8615

* Recall: 0.9155

* F1: 0.8877

* AUC: 0.9320

1) Model LR L2 (Ridge), C = 7.0 Won in all three strategies, keeps all the features while shrinking all irrelevant words. C = 7.0 was the best C that did not add too much regularization but not too little.

2) Validation Stratified K-Fold with K=10. It was the most reliable estimate due to that it kept the balance of 62% positive reviews and 38% negative reviews in every fold. It had the lowest std at K = 10 between the L1 and L2 and it uses all 2000 samples.






# Text Rollout X Predictions

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


# Load training data and rollout file
df_train_answer   = pd.read_csv('/content/drive/MyDrive/text_training.csv')        # adjust path as needed
df_rollout_answer = pd.read_csv('/content/drive/MyDrive/text_rollout_X.csv')       # adjust path as needed

#Step 2: Prepare training data
X_train = df_train_answer.drop(columns=['ID', 'rating'])
y_train = df_train_answer['rating']

# Prepare rollout data (keep ID, drop it from features)
rollout_ids = df_rollout_answer['ID']
X_rollout   = df_rollout_answer.drop(columns=['ID','rating'],errors='ignore')

#TF-IDF — fit on ALL training data, transform both
tfidf = TfidfTransformer()
X_train_tfidf   = tfidf.fit_transform(X_train)
X_rollout_tfidf = tfidf.transform(X_rollout)

# Train final model on all 2000 samples
final_model = LogisticRegression(penalty='l2', solver='liblinear',
                                 C=7.0, max_iter=1000, random_state=42)
final_model.fit(X_train_tfidf, y_train)

# Predict
predictions = final_model.predict(X_rollout_tfidf)

#Save results
results = pd.DataFrame({
    'ID':     rollout_ids,
    'rating': predictions
})

results.to_csv('Ariel_Sobol_311116867_Guy_Berelzon_203865084 .csv', index=False)
print(results.head(10))
print(f"\nTotal predictions: {len(results)}")
print(f"Positive (1): {sum(predictions == 1)}")
print(f"Negative (0): {sum(predictions == 0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ID  rating
0  2001       0
1  2002       1
2  2003       1
3  2004       1
4  2005       1
5  2006       1
6  2007       1
7  2008       1
8  2009       1
9  2010       1

Total predictions: 2000
Positive (1): 1303
Negative (0): 697
